# 04 — Severity Benchmark: Classical CV vs Fine-tuned CNN

Both severity methods are evaluated on the held-out **test set**.

| Method | How | Extra training? |
|---|---|---|
| Classical CV | Edge density + depth score + texture score → weighted sum | No |
| Fine-tuned CNN | YOLOv8n-cls on auto-labelled crops | Yes (03 notebook) |

The winner is written to `configs/pipeline_config.yaml` and used at runtime.

**Speed note:** CNN inference uses **batched evaluation** (not 1-image-at-a-time).

In [ ]:
import sys, time
sys.path.insert(0, '../src')   # single insert is sufficient

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, f1_score

BASE_DIR      = Path('..').resolve()
SEVERITY_TEST = BASE_DIR / 'data' / 'processed' / 'severity_crops' / 'test'
MODEL_PATH    = BASE_DIR / 'models' / 'severity_model.pt'

from analyzers.pothole_analyzer import _compute_cv_scores, _cv_severity

CLASSES          = ['Low', 'Medium', 'High']
CNN_IDX_TO_LABEL = {0: 'High', 1: 'Low', 2: 'Medium'}   # YOLOv8-cls alphabetical order
BATCH_SIZE       = 64

## 1. Load Test Set

In [ ]:
assert SEVERITY_TEST.exists(), 'Run python src/prepare_data.py first'

test_imgs, y_true = [], []
for cls in CLASSES:
    for p in sorted((SEVERITY_TEST / cls).glob('*')):
        if p.suffix.lower() not in ('.jpg', '.jpeg', '.png'): continue
        img = cv2.imread(str(p))
        if img is not None:
            test_imgs.append(img)
            y_true.append(cls)

print(f'Test set: {len(test_imgs)} crops')
print('Distribution:', dict(Counter(y_true)))

## 2. Classical CV — Evaluate

In [ ]:
t0 = time.time()
cv_preds      = []
cv_scores_all = []
for img in test_imgs:
    scores = _compute_cv_scores(img)
    cv_scores_all.append(scores)
    cv_preds.append(_cv_severity(scores))
cv_time = time.time() - t0

cv_acc = sum(t == p for t, p in zip(y_true, cv_preds)) / len(y_true)
print(f'Classical CV')
print(f'  Accuracy : {cv_acc:.4f} ({cv_acc*100:.1f}%)')
print(f'  Time     : {cv_time:.3f}s  ({len(test_imgs)/cv_time:.0f} crops/sec)')

## 3. CNN — Batched Evaluation

In [ ]:
cnn_preds = None
cnn_acc   = None
cnn_time  = None

if MODEL_PATH.exists():
    from ultralytics import YOLO
    cnn_model = YOLO(str(MODEL_PATH))

    t0        = time.time()
    cnn_preds = []
    for i in range(0, len(test_imgs), BATCH_SIZE):
        batch   = test_imgs[i : i + BATCH_SIZE]
        results = cnn_model(batch, imgsz=128, verbose=False)
        for r in results:
            cnn_preds.append(CNN_IDX_TO_LABEL[int(r.probs.top1)])
    cnn_time = time.time() - t0

    cnn_acc = sum(t == p for t, p in zip(y_true, cnn_preds)) / len(y_true)
    print(f'Fine-tuned CNN (batch={BATCH_SIZE})')
    print(f'  Accuracy : {cnn_acc:.4f} ({cnn_acc*100:.1f}%)')
    print(f'  Time     : {cnn_time:.3f}s  ({len(test_imgs)/cnn_time:.0f} crops/sec)')
else:
    print(f'CNN model not found at {MODEL_PATH}')
    print('Run 03-train-severity.ipynb first')

## 4. Accuracy + Speed Comparison

In [ ]:
methods  = ['Classical CV']
accs     = [cv_acc]
times    = [cv_time]
clrs     = ['#3498db']

if cnn_acc is not None:
    methods.append('Fine-tuned CNN')
    accs.append(cnn_acc)
    times.append(cnn_time)
    clrs.append('#e74c3c')

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Accuracy
bars = axes[0].bar(methods, accs, color=clrs, width=0.4)
winner_idx = accs.index(max(accs))
bars[winner_idx].set_edgecolor('gold'); bars[winner_idx].set_linewidth(3)
for bar, acc in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{acc:.2%}', ha='center', fontsize=12, fontweight='bold')
axes[0].set_ylim(0, 1.15)
axes[0].set_title('Accuracy (higher is better)')
axes[0].set_ylabel('Accuracy')
axes[0].axhline(1/3, color='gray', linestyle=':', alpha=0.5, label='Random baseline')
axes[0].legend(fontsize=8)

# Speed (crops/sec)
speeds = [len(test_imgs)/t for t in times]
axes[1].bar(methods, speeds, color=clrs, width=0.4)
for i, (bar, s) in enumerate(zip(axes[1].patches, speeds)):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'{s:.0f}/s', ha='center', fontsize=11, fontweight='bold')
axes[1].set_title('Inference Speed (higher is better)')
axes[1].set_ylabel('Crops per second')

plt.suptitle('Severity Method Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nWINNER (accuracy): {methods[winner_idx]} → {accs[winner_idx]:.2%}')

## 5. Side-by-Side Confusion Matrices

In [ ]:
n_cols = 1 + (1 if cnn_preds else 0)
fig, axes = plt.subplots(1, n_cols, figsize=(6 * n_cols, 5))
if n_cols == 1: axes = [axes]

pairs = [(cv_preds, f'Classical CV ({cv_acc:.2%})')]
if cnn_preds:
    pairs.append((cnn_preds, f'Fine-tuned CNN ({cnn_acc:.2%})'))

for ax, (preds, title) in zip(axes, pairs):
    cm   = confusion_matrix(y_true, preds, labels=CLASSES)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title, fontsize=12)

plt.suptitle('Confusion Matrices — Severity Benchmark', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Per-Class F1 Comparison

In [ ]:
cv_f1  = f1_score(y_true, cv_preds, labels=CLASSES, average=None)
x      = np.arange(len(CLASSES))
width  = 0.35

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - width/2 if cnn_preds else x, cv_f1, width, label='Classical CV', color='#3498db')

if cnn_preds:
    cnn_f1 = f1_score(y_true, cnn_preds, labels=CLASSES, average=None)
    ax.bar(x + width/2, cnn_f1, width, label='Fine-tuned CNN', color='#e74c3c')

ax.set_xticks(x); ax.set_xticklabels(CLASSES)
ax.set_ylim(0, 1.15); ax.set_ylabel('F1 Score')
ax.set_title('Per-Class F1 — CV vs CNN')
ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print('CV  classification report:')
print(classification_report(y_true, cv_preds, target_names=CLASSES))
if cnn_preds:
    print('CNN classification report:')
    print(classification_report(y_true, cnn_preds, target_names=CLASSES))

## 7. Examples: Where Methods Disagree

In [ ]:
if cnn_preds:
    cases = {
        'CNN correct / CV wrong':  [(i, img) for i, (img, t) in enumerate(zip(test_imgs, y_true))
                                    if cnn_preds[i] == t and cv_preds[i] != t][:4],
        'CV correct / CNN wrong':  [(i, img) for i, (img, t) in enumerate(zip(test_imgs, y_true))
                                    if cv_preds[i] == t and cnn_preds[i] != t][:4],
    }
    for title, items in cases.items():
        if not items:
            print(f'No cases: {title}'); continue
        fig, axes = plt.subplots(1, len(items), figsize=(3*len(items), 3))
        if len(items) == 1: axes = [axes]
        for ax, (idx, img) in zip(axes, items):
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            ax.axis('off')
            ax.set_title(f'True: {y_true[idx]}\nCV: {cv_preds[idx]}\nCNN: {cnn_preds[idx]}',
                         fontsize=8)
        plt.suptitle(title, fontsize=11)
        plt.tight_layout(); plt.show()
else:
    print('CNN model not available for disagreement analysis.')

## 8. Write Winner to Pipeline Config

In [ ]:
import yaml

winner = methods[winner_idx].lower().replace('classical cv', 'cv').replace('fine-tuned cnn', 'cnn')

CONFIG_PATH = BASE_DIR / 'configs' / 'pipeline_config.yaml'
CONFIG_PATH.parent.mkdir(exist_ok=True)

if CONFIG_PATH.exists():
    cfg = yaml.safe_load(CONFIG_PATH.read_text()) or {}
else:
    cfg = {}

cfg['severity_method'] = winner
CONFIG_PATH.write_text(yaml.dump(cfg, default_flow_style=False))

print(f'Pipeline config updated:')
print(f'  severity_method: {winner}')
print(f'  File: {CONFIG_PATH}')
print()
print('Next: run 05-pipeline-demo.ipynb')